# Model 3: Faster R-CNN (Detection)\n\nTwo-stage object detector using ResNet50-FPN v2 backbone. Detection-only — no native masks. Watershed is applied afterward as post-processing to generate precise segmentation masks.\n\n**Architecture**: ResNet50-FPN v2 → RPN → ROI Align → Box Head (classification + regression)\n**Training**: SGD with cosine annealing, mixed precision, gradient accumulation

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import os, json, numpy as np, cv2, time
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

BASE_DIR    = r"C:\Users\User\Desktop\Ipynb"
DATASET_DIR = os.path.join(BASE_DIR, "archive", "dataset_v2")
OUTPUT_DIR  = os.path.join(BASE_DIR, "runs", "faster_rcnn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CLASSES = 4    # 3 waste categories (Recyclable, Non-recyclable, Other) + background
CLASS_NAMES = ["background", "Recyclable", "Non-recyclable", "Other"]
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset & DataLoaders

In [ ]:
import albumentations as A
from torch.utils.data import Dataset, DataLoader


class TACODetectionDataset(Dataset):
    """TACO dataset for Faster R-CNN — bounding boxes + class labels only, no masks."""

    def __init__(self, annotation_file, image_dir, transforms=None, max_size=1024):
        self.image_dir  = image_dir
        self.transforms = transforms
        self.max_size   = max_size

        with open(annotation_file) as f:
            data = json.load(f)

        self.ann_by_image = defaultdict(list)
        for ann in data["annotations"]:
            self.ann_by_image[ann["image_id"]].append(ann)

        self.valid_images = [
            img for img in data["images"]
            if os.path.exists(os.path.join(image_dir, img["file_name"]))
        ]
        print(f"  {len(self.valid_images)} images, "
              f"{sum(len(self.ann_by_image[i['id']]) for i in self.valid_images)} annotations")

    def __len__(self):
        return len(self.valid_images)

    def __getitem__(self, idx):
        img_info = self.valid_images[idx]
        image    = cv2.imread(os.path.join(self.image_dir, img_info["file_name"]))
        image    = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w     = image.shape[:2]

        scale = min(self.max_size / max(h, w), 1.0)
        if scale < 1.0:
            image = cv2.resize(image, (int(w * scale), int(h * scale)))
            h, w  = image.shape[:2]

        anns   = self.ann_by_image.get(img_info["id"], [])
        boxes, labels = [], []

        for ann in anns:
            bx, by, bw, bh = [v * scale for v in ann["bbox"]]
            if bw < 2 or bh < 2:
                continue
            x1 = max(0.0, bx);           y1 = max(0.0, by)
            x2 = min(float(w), bx + bw); y2 = min(float(h), by + bh)
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            boxes.append([x1, y1, x2, y2])
            labels.append(ann["category_id"])

        if self.transforms and boxes:
            t = self.transforms(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = t["image"], t["bboxes"], t["labels"]

        img_t = torch.as_tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0

        if not boxes:
            target = {
                "boxes":    torch.zeros((0, 4), dtype=torch.float32),
                "labels":   torch.zeros((0,),   dtype=torch.int64),
                "area":     torch.zeros((0,),   dtype=torch.float32),
                "iscrowd":  torch.zeros((0,),   dtype=torch.int64),
                "image_id": torch.tensor(img_info["id"]),
            }
        else:
            b = torch.as_tensor(boxes, dtype=torch.float32)
            target = {
                "boxes":    b,
                "labels":   torch.as_tensor(labels, dtype=torch.int64),
                "area":     (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]),
                "iscrowd":  torch.zeros((len(boxes),), dtype=torch.int64),
                "image_id": torch.tensor(img_info["id"]),
            }
        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.RandomRotate90(p=0.2),
], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"],
                             min_area=100, min_visibility=0.3))

print("Loading datasets…")
train_ds = TACODetectionDataset(os.path.join(DATASET_DIR, "train_annotations.json"),
                                 os.path.join(DATASET_DIR, "images", "train"), train_tf)
val_ds   = TACODetectionDataset(os.path.join(DATASET_DIR, "val_annotations.json"),
                                 os.path.join(DATASET_DIR, "images", "val"))
test_ds  = TACODetectionDataset(os.path.join(DATASET_DIR, "test_annotations.json"),
                                 os.path.join(DATASET_DIR, "images", "test"))

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## Build Model & Train

In [ ]:
# Build Faster R-CNN
model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total/1e6:.1f}M")

# Training config
NUM_EPOCHS       = 50
ACCUM_STEPS      = 2        # effective batch = 4 * 2 = 8
PATIENCE         = 15

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
scaler    = torch.amp.GradScaler("cuda")

train_losses, val_losses = [], []
best_val_loss, patience_ctr = float("inf"), 0

for epoch in range(NUM_EPOCHS):
    # --- train ---
    model.train()
    ep_loss, nb = 0.0, 0
    optimizer.zero_grad()
    for i, (imgs, tgts) in enumerate(tqdm(train_loader, desc=f"Ep{epoch+1} Train")):
        imgs = [x.to(DEVICE) for x in imgs]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
        with torch.amp.autocast("cuda"):
            loss_dict = model(imgs, tgts)
            loss = sum(loss_dict.values()) / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        ep_loss += loss.item() * ACCUM_STEPS; nb += 1
    train_losses.append(ep_loss / nb)

    # --- val ---
    model.train()
    vl, vb = 0.0, 0
    with torch.no_grad():
        for imgs, tgts in tqdm(val_loader, desc=f"Ep{epoch+1} Val"):
            imgs = [x.to(DEVICE) for x in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            with torch.amp.autocast("cuda"):
                loss_dict = model(imgs, tgts)
                vl += sum(loss_dict.values()).item(); vb += 1
    val_losses.append(vl / vb)
    scheduler.step(epoch)
    print(f"  Ep {epoch+1}: train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}  lr={optimizer.param_groups[0]['lr']:.6f}")

    if val_losses[-1] < best_val_loss:
        best_val_loss = val_losses[-1]; patience_ctr = 0
        torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(),
                    "val_loss": best_val_loss}, os.path.join(OUTPUT_DIR, "best_faster_rcnn.pth"))
        print(f"  -> best saved (val_loss={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}"); break

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label="Train"); ax.plot(val_losses, label="Val")
ax.set(xlabel="Epoch", ylabel="Loss", title="Faster R-CNN Training")
ax.legend(); ax.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()
print("Training complete.")

## Test Evaluation

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Load best checkpoint
ckpt = torch.load(os.path.join(OUTPUT_DIR, "best_faster_rcnn.pth"), map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

# Load test annotations
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    test_coco_data = json.load(f)

gt_counts      = defaultdict(int)
for ann in test_coco_data["annotations"]:
    gt_counts[ann["image_id"]] += 1

img_info_by_id = {img["id"]: img for img in test_coco_data["images"]}

coco_preds        = []
per_image_results = []
count_errors      = []
inference_times   = []

CONF_THRESHOLD = 0.5

with torch.no_grad():
    for imgs, tgts in tqdm(test_loader, desc="Test inference"):
        img_t  = imgs[0]
        tgt    = tgts[0]
        img_id = tgt["image_id"].item()

        # Original image dimensions — needed to undo the dataset's resize
        img_info        = img_info_by_id[img_id]
        orig_h, orig_w  = img_info["height"], img_info["width"]
        scale           = min(1024 / max(orig_h, orig_w), 1.0)

        t0 = time.perf_counter()
        with torch.amp.autocast("cuda"):
            out = model([img_t.to(DEVICE)])[0]
        inf_ms = (time.perf_counter() - t0) * 1000
        inference_times.append(inf_ms)

        keep   = out["scores"] >= CONF_THRESHOLD
        boxes  = out["boxes"][keep].cpu()
        scores = out["scores"][keep].cpu()
        labels = out["labels"][keep].cpu()

        gt_count   = gt_counts.get(img_id, 0)
        pred_count = len(boxes)
        count_errors.append(abs(pred_count - gt_count))

        fname = img_info["file_name"]
        per_image_results.append({
            "file":         fname,
            "gt_count":     gt_count,
            "pred_count":   pred_count,
            "count_error":  abs(pred_count - gt_count),
            "inference_ms": inf_ms,
        })

        # Boxes are in resized (max_size=1024) pixel space;
        # divide by scale to convert back to original image coordinates.
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = [v / scale for v in box.tolist()]
            coco_preds.append({
                "image_id":    img_id,
                "category_id": int(label),
                "bbox":        [x1, y1, x2 - x1, y2 - y1],
                "score":       float(score),
            })

# COCO bbox evaluation
gt_coco = COCO()
gt_coco.dataset = test_coco_data
gt_coco.createIndex()

box_map50 = box_map50_95 = 0.0
if coco_preds:
    dt       = gt_coco.loadRes(coco_preds)
    ev       = COCOeval(gt_coco, dt, "bbox")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    box_map50    = float(ev.stats[1])
    box_map50_95 = float(ev.stats[0])
else:
    print("No predictions above threshold.")

within_1 = sum(1 for e in count_errors if e <= 1) / len(count_errors) * 100
within_3 = sum(1 for e in count_errors if e <= 3) / len(count_errors) * 100
avg_inf  = float(np.mean(inference_times[1:]) if len(inference_times) > 1 else inference_times[0])

print(f"\n=== Faster R-CNN Test Results ===")
print(f"  Box mAP@50:       {box_map50:.4f}")
print(f"  Box mAP@50-95:    {box_map50_95:.4f}")
print(f"  Counting MAE:     {np.mean(count_errors):.2f}")
print(f"  Within +/-1:      {within_1:.1f}%")
print(f"  Within +/-3:      {within_3:.1f}%")
print(f"  Inf speed:        {avg_inf:.1f} ms/image")

results_data = {
    "metrics": {
        "box_map50":         box_map50,
        "box_map50_95":      box_map50_95,
        "mask_map50":        0.0,
        "mask_map50_95":     0.0,
        "counting_mae":      float(np.mean(count_errors)),
        "counting_within_1": within_1,
        "counting_within_3": within_3,
        "avg_inference_ms":  avg_inf,
    },
    "per_image": per_image_results,
}
with open(os.path.join(OUTPUT_DIR, "faster_rcnn_results.json"), "w") as f:
    json.dump(results_data, f, indent=2)
print("Saved faster_rcnn_results.json")

## Visualize Predictions

In [ ]:
COLORS = plt.colormaps["tab10"]

def draw_boxes(ax, image, boxes, labels, scores, conf=0.25):
    ax.imshow(image)
    for box, label, score in zip(boxes, labels, scores):
        if score < conf:
            continue
        x1, y1, x2, y2 = box
        c = COLORS(label % NUM_CLASSES)
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                              linewidth=2, edgecolor=c, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{CLASS_NAMES[label]} {score:.2f}",
                color="white", fontsize=7, bbox=dict(facecolor=c, alpha=0.7, pad=1))
    ax.axis("off")

model.eval()
samples = []
for imgs, tgts in test_loader:
    samples.append((imgs[0], tgts[0]))
    if len(samples) == 8:
        break

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, (img_t, tgt) in zip(axes.flat, samples):
    img_np = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    with torch.no_grad():
        with torch.amp.autocast("cuda"):
            out = model([img_t.to(DEVICE)])[0]
    boxes  = out["boxes"].cpu().numpy()
    labels = out["labels"].cpu().numpy()
    scores = out["scores"].cpu().numpy()
    draw_boxes(ax, img_np, boxes, labels, scores)
    ax.set_title(f"GT:{len(tgt['boxes'])} Pred:{(scores>=0.25).sum()}", fontsize=9)

plt.suptitle("Faster R-CNN Predictions (conf≥0.25)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=150)
plt.show()
print("Saved sample_predictions.png")

## Algorithm 3: Watershed Post-Processing

Faster R-CNN provides bounding boxes with class labels. Watershed refines each box crop into a pixel-precise segmentation mask.

```
Trained Faster R-CNN → boxes + class labels
  → For each box: crop image (+ padding) → watershed_on_crop()
  → Place mask back on full-image canvas
  → Evaluate: box mAP + mask mAP + binary IoU + counting
```

In [ ]:
# ── Watershed helpers ─────────────────────────────────────────────────────────
from pycocotools import mask as coco_mask_util
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def watershed_on_crop(crop, padding=10):
    """Run watershed on a single-object image crop. Returns binary mask (0/255)."""
    h, w     = crop.shape[:2]
    gray     = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    filtered = cv2.bilateralFilter(crop, 9, 75, 75)
    a_chan   = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)[:, :, 1]

    _, thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh_a = cv2.adaptiveThreshold(a_chan, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 15, 2)
    edges    = cv2.Canny(gray, 30, 100)
    combined = cv2.bitwise_or(thresh_otsu, thresh_a)
    combined = cv2.add(combined, cv2.dilate(edges, np.ones((2, 2), np.uint8)))
    _, combined = cv2.threshold(combined, 127, 255, cv2.THRESH_BINARY)

    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(combined, cv2.MORPH_OPEN,  kernel, iterations=1)
    cleaned = cv2.morphologyEx(cleaned,  cv2.MORPH_CLOSE, kernel, iterations=2)

    bw = max(3, min(padding // 2, 8))
    border_mask = np.ones((h, w), dtype=np.uint8) * 255
    border_mask[:bw, :] = border_mask[-bw:, :] = 0
    border_mask[:, :bw] = border_mask[:, -bw:] = 0
    sure_bg = cv2.bitwise_or(cv2.dilate(cleaned, kernel, iterations=3), border_mask)

    dist = cv2.distanceTransform(cleaned, cv2.DIST_L2, 5)
    if dist.max() > 0:
        _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
        sure_fg = np.uint8(sure_fg)
    else:
        sure_fg = np.zeros((h, w), dtype=np.uint8)
        cx, cy  = w // 2, h // 2
        r = min(cx, cy, w - cx, h - cy) // 2
        if r > 0:
            cv2.circle(sure_fg, (cx, cy), r, 255, -1)

    unknown  = cv2.subtract(sure_bg, sure_fg)
    _, marks = cv2.connectedComponents(sure_fg)
    marks    = marks + 1
    marks[unknown == 255] = 0
    marks    = cv2.watershed(crop, marks)

    mask  = np.zeros((h, w), dtype=np.uint8)
    valid = set(np.unique(marks)) - {-1, 1}
    if valid:
        best = max(valid, key=lambda lbl: np.sum(marks == lbl))
        mask[marks == best] = 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    else:
        mask = cleaned
    return mask


def apply_watershed_to_boxes(image, boxes, labels, scores, conf=0.25, padding=15):
    """Apply watershed on each detected box crop. Returns list of detection dicts."""
    h, w = image.shape[:2]
    dets = []
    for box, label, score in zip(boxes, labels, scores):
        if score < conf:
            continue
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        x1c = max(0, x1 - padding); y1c = max(0, y1 - padding)
        x2c = min(w, x2 + padding); y2c = min(h, y2 + padding)
        if x2c - x1c < 5 or y2c - y1c < 5:
            continue
        crop_mask = watershed_on_crop(image[y1c:y2c, x1c:x2c], padding=padding)
        full_mask = np.zeros((h, w), dtype=np.uint8)
        full_mask[y1c:y2c, x1c:x2c] = crop_mask
        dets.append({"box": [x1, y1, x2, y2], "class_id": int(label),
                     "score": float(score), "mask": full_mask})
    return dets


def build_gt_mask_ws(anns, img_shape):
    combined = np.zeros(img_shape[:2], dtype=np.uint8)
    for ann in anns:
        for seg in ann.get("segmentation", []):
            pts = np.array(seg, dtype=np.float32).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(combined, [pts], 255)
    return combined


def evaluate_watershed_detections(det_by_image, coco_data, img_id_lookup):
    """Compute box mAP, mask mAP (watershed), binary IoU, and counting metrics."""
    gt_counts = defaultdict(int)
    gt_masks  = defaultdict(list)
    for ann in coco_data["annotations"]:
        gt_counts[ann["image_id"]] += 1
        gt_masks[ann["image_id"]].append(ann)

    bbox_preds, segm_preds = [], []
    binary_ious, count_errs, per_image = [], [], []

    for image_id, (img_shape, fname, dets) in det_by_image.items():
        info   = img_id_lookup.get(image_id, {})
        orig_h = info.get("height", img_shape[0])
        orig_w = info.get("width",  img_shape[1])

        pred_m = np.zeros(img_shape[:2], dtype=np.uint8)
        for d in dets:
            pred_m = cv2.bitwise_or(pred_m, d["mask"])
        gt_m   = build_gt_mask_ws(gt_masks.get(image_id, []), img_shape)
        inter  = cv2.countNonZero(cv2.bitwise_and(pred_m, gt_m))
        union  = cv2.countNonZero(cv2.bitwise_or( pred_m, gt_m))
        binary_ious.append(inter / max(union, 1))

        gt_count = gt_counts.get(image_id, 0)
        count_errs.append(abs(len(dets) - gt_count))
        per_image.append({"file": fname, "gt_count": gt_count,
                          "pred_count": len(dets), "count_error": abs(len(dets) - gt_count)})

        for d in dets:
            x1, y1, x2, y2 = d["box"]
            bbox_preds.append({"image_id": image_id, "category_id": d["class_id"],
                               "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                               "score": d["score"]})
            m = d["mask"].copy()
            if m.shape != (orig_h, orig_w):
                m = cv2.resize(m, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            rle = coco_mask_util.encode(np.asfortranarray(m))
            rle["counts"] = rle["counts"].decode("utf-8")
            segm_preds.append({"image_id": image_id, "category_id": d["class_id"],
                               "segmentation": rle, "score": d["score"]})

    gt_coco = COCO(); gt_coco.dataset = coco_data; gt_coco.createIndex()
    b50 = b95 = m50 = m95 = 0.0
    if bbox_preds:
        dt = gt_coco.loadRes(bbox_preds)
        ev = COCOeval(gt_coco, dt, "bbox"); ev.evaluate(); ev.accumulate(); ev.summarize()
        b50, b95 = float(ev.stats[1]), float(ev.stats[0])
    if segm_preds:
        dt = gt_coco.loadRes(segm_preds)
        ev = COCOeval(gt_coco, dt, "segm"); ev.evaluate(); ev.accumulate(); ev.summarize()
        m50, m95 = float(ev.stats[1]), float(ev.stats[0])

    n = len(count_errs)
    metrics = {
        "box_map50":         b50,  "box_map50_95":       b95,
        "mask_map50":        m50,  "mask_map50_95":      m95,
        "binary_iou":        float(np.mean(binary_ious)),
        "counting_mae":      float(np.mean(count_errs)),
        "counting_within_1": float(sum(1 for e in count_errs if e <= 1) / n * 100),
        "counting_within_3": float(sum(1 for e in count_errs if e <= 3) / n * 100),
    }
    return metrics, per_image


print("Watershed helpers loaded.")

In [ ]:
CONF_WS    = 0.25
PADDING_WS = 15
MAX_SIZE   = 1024

# Build test-image lookups (test_coco_data already loaded above)
wt_img_file_lookup = {img["file_name"]: img for img in test_coco_data["images"]}
wt_img_id_lookup   = {img["id"]: img for img in test_coco_data["images"]}
wt_test_image_dir  = os.path.join(DATASET_DIR, "images", "test")
wt_test_files      = sorted([
    f for f in os.listdir(wt_test_image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

# Reload best checkpoint to ensure we use best weights
ckpt = torch.load(os.path.join(OUTPUT_DIR, "best_faster_rcnn.pth"), map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Checkpoint epoch {ckpt['epoch']}  val_loss={ckpt['val_loss']:.4f}")

det_by_image = {}
det_times    = []

with torch.no_grad():
    for fname in tqdm(wt_test_files, desc="FRCNN+Watershed"):
        img_bgr  = cv2.imread(os.path.join(wt_test_image_dir, fname))
        if img_bgr is None:
            continue
        img_info = wt_img_file_lookup.get(fname)
        if img_info is None:
            continue
        image_id       = img_info["id"]
        h_orig, w_orig = img_bgr.shape[:2]

        scale   = min(MAX_SIZE / max(h_orig, w_orig), 1.0)
        img_res = cv2.resize(img_bgr, (int(w_orig * scale), int(h_orig * scale))) if scale < 1.0 else img_bgr
        img_rgb = cv2.cvtColor(img_res, cv2.COLOR_BGR2RGB)
        img_t   = torch.as_tensor(img_rgb, dtype=torch.float32).permute(2, 0, 1) / 255.0

        t0 = time.perf_counter()
        with torch.amp.autocast("cuda"):
            out = model([img_t.to(DEVICE)])[0]
        det_times.append((time.perf_counter() - t0) * 1000)

        boxes  = out["boxes"].cpu().numpy() / scale
        labels = out["labels"].cpu().numpy()
        scores = out["scores"].cpu().numpy()

        dets = apply_watershed_to_boxes(img_bgr, boxes, labels, scores, CONF_WS, PADDING_WS)
        det_by_image[image_id] = (img_bgr.shape, fname, dets)

print("\nEvaluating…")
metrics, per_image = evaluate_watershed_detections(det_by_image, test_coco_data, wt_img_id_lookup)
avg_inf = float(np.mean(det_times[1:]) if len(det_times) > 1 else det_times[0])

print(f"\n{'='*50}")
print("Algorithm 3: Faster R-CNN + Watershed")
print(f"{'='*50}")
for k, v in metrics.items():
    print(f"  {k:25s}: {v:.4f}")
print(f"  {'avg_inference_ms':25s}: {avg_inf:.1f}")

results_data = {
    "metrics": {**metrics, "avg_inference_ms": avg_inf},
    "per_image": per_image,
}
out_path = os.path.join(OUTPUT_DIR, "fasterrcnn_watershed_results.json")
with open(out_path, "w") as f:
    json.dump(results_data, f, indent=2)
print(f"\nSaved: {out_path}")